In [ ]:
import pandas as pd
import glob
import os
import datetime
from tqdm import tqdm
import numpy as np
import re
from IPython.display import display

pd.set_option('future.no_silent_downcasting', True)

PATH_1 = "data/master_registry.xlsx"
PATH_2 = "data/reference_dictionary.parquet"
DIR_INPUT = "data/weekly_updates/"
PATH_3 = "data/raw_logs/"

def func_clean_cols(df):
    df.columns = df.columns.str.upper().str.strip()
    for col in df.select_dtypes(include=[object]).columns:
        df[col] = df[col].map(lambda x: x.upper().strip() if isinstance(x, str) else x)
    return df

def func_clean_id(val):
    try:
        if pd.isna(val) or str(val).strip().upper() in ['NAN', 'NONE', '']:
            return np.nan
        return str(int(float(val)))
    except (ValueError, TypeError):
        return str(val).strip()

def func_get_status(val):
    if pd.isna(val): return 'GRAY'
    if val == 0: return '⚫ BLACK'
    if 0 < val < 49: return "🔴 RED"
    if 49 <= val < 80: return "🟡 YELLOW"
    if 80 <= val <= 100: return "🟢 GREEN"
    return "🔵 BLUE"

df_1 = pd.read_excel(PATH_1)
df_1 = func_clean_cols(df_1)
df_1 = df_1[(df_1['CATEGORY_TYPE'] == 'PRIMARY') & (df_1['REGION'] == 'REGION_A')].copy()
df_1 = df_1[df_1['LOG_CODE'] != 'EXCLUDE_001']
list_target_codes = df_1['LOG_CODE'].unique().tolist()

raw_files = glob.glob(os.path.join(PATH_3, "*.xls*"))
list_dfs_raw = []

for f in raw_files:
    try:
        temp_df = pd.read_excel(f)
        temp_df['SOURCE_FILE'] = os.path.basename(f)
        list_dfs_raw.append(temp_df)
    except Exception as e:
        pass

if list_dfs_raw:
    df_2 = pd.concat(list_dfs_raw, ignore_index=True)
else:
    df_2 = pd.DataFrame()

df_2 = func_clean_cols(df_2)
df_2['USER_ID'] = df_2['USER_ID'].astype(float).astype('Int64').astype(str)
df_2['DEVICE_IMEI'] = df_2['DEVICE_IMEI'].astype(float).astype('Int64').astype(str)
df_2['TIMESTAMP'] = pd.to_datetime(df_2['TIMESTAMP'], dayfirst=True)
df_2['WEEK_NUM'] = df_2['TIMESTAMP'].dt.isocalendar().week.astype(int)
list_weeks = sorted(df_2['WEEK_NUM'].unique().tolist())

df_3 = pd.read_parquet(PATH_2)
df_3 = func_clean_cols(df_3)
df_3['CONTRACT_ID'] = df_3['CONTRACT_ID'].apply(func_clean_id)
df_3 = df_3[df_3['CATEGORY_TYPE'] == 'PRIMARY'].copy()

report_summary = []
list_all_inputs = []
list_analysis_processed = []
list_missing_logs = []
list_unmatched_logic = []
list_out_of_target = []
list_global_out_logs = []
list_unknown_entities = []
list_not_in_dict = []

for wk in list_weeks:
    target_input_path = None
    input_files = glob.glob(os.path.join(DIR_INPUT, "*.xlsx"))
    for f in input_files:
        if re.match(rf"^{wk}_", os.path.basename(f)):
            target_input_path = f
            break
            
    if not target_input_path:
        continue
        
    current_filename = os.path.basename(target_input_path)
    file_week_id = current_filename.split('_')[0]
    df_4 = pd.read_excel(target_input_path, header=1)
    df_4.columns = df_4.columns.str.upper().str.strip()
    df_4['FILE_WEEK_REF'] = file_week_id
    
    mapping_schema = {
        'ID_REF': 'CONTRACT_ID', 'TIME_SLOT':'TIME_SLOT', 'ENTITY_NAME': 'ENTITY_NAME',
        'OBJECT_TYPE': 'OBJECT_TYPE', 'ZONE': 'ZONE', 
        'AUTH_TOKEN': 'AUTH_TOKEN', 'TARGET_FREQ': 'WEEKLY_FREQUENCY',
        'FILE_WEEK_REF': 'FILE_WEEK_REF'  
    }
    
    active_cols = [c for c in mapping_schema.keys() if c in df_4.columns]
    df_4 = df_4[active_cols].rename(columns=mapping_schema).copy()
    df_4 = df_4.map(lambda x: x.strip() if isinstance(x, str) else x)
    
    filt_1 = (df_4['AUTH_TOKEN'].isna()) | (df_4['AUTH_TOKEN'] == '')
    filt_2 = (df_4['WEEKLY_FREQUENCY'].notna()) & (df_4['WEEKLY_FREQUENCY'] != 'SPECIAL_STATUS')
    df_4 = df_4[filt_1 & filt_2].copy()
    
    df_4['CONTRACT_ID'] = df_4['CONTRACT_ID'].apply(func_clean_id)
    null_mask = df_4['CONTRACT_ID'].isna()
    
    if null_mask.any():
        placeholder_ids = [f"UNKNOWN_WK{wk}_{i}" for i in range(null_mask.sum())]
        df_4.loc[null_mask, 'CONTRACT_ID'] = placeholder_ids

    df_4_history = df_4.copy()
    df_4_history['WEEK_NUM'] = wk
    df_4_history['WEEKLY_FREQUENCY'] = df_4_history['WEEKLY_FREQUENCY'].astype(str).str.split('/').str[0]
    df_4_history['WEEKLY_FREQUENCY'] = pd.to_numeric(df_4_history['WEEKLY_FREQUENCY'], errors='coerce').fillna(0).astype(int)
    list_all_inputs.append(df_4_history)
    
    df_4 = df_4.drop_duplicates(subset=['CONTRACT_ID']).copy()
    total_expected = len(df_4)
    unknowns = [c for c in df_4['CONTRACT_ID'] if str(c).startswith('UNKNOWN')]
    df_unknown_tmp = df_4[df_4['CONTRACT_ID'].isin(unknowns)].copy()
    df_unknown_tmp['WEEK_NUM'] = wk
    list_unknown_entities.append(df_unknown_tmp)
    
    df_4_3 = pd.merge(df_4, df_3, on='CONTRACT_ID', how='left')
    in_dict_list = df_4_3[df_4_3['TAG_ID'].notna()]['CONTRACT_ID'].unique().tolist()
    
    missing_from_dict = [c for c in df_4['CONTRACT_ID'] if c not in in_dict_list and c not in unknowns]
    df_missing_dict_tmp = df_4[df_4['CONTRACT_ID'].isin(missing_from_dict)].copy()
    df_missing_dict_tmp['WEEK_NUM'] = wk
    list_not_in_dict.append(df_missing_dict_tmp)
    
    df_2_wk = df_2[df_2['WEEK_NUM'] == wk]
    df_4_valid = df_4_3.dropna(subset=['TAG_ID']).copy()
    df_processed = pd.merge(df_4_valid, df_2_wk, left_on='TAG_ID', right_on='LOGGED_TAG', how='inner')
    
    df_processed['IS_TARGET'] = df_processed['LOG_CODE'].isin(list_target_codes)
    ids_in_target = df_processed[df_processed['IS_TARGET']]['CONTRACT_ID'].unique().tolist()
    ids_out_all = df_processed[~df_processed['IS_TARGET']]['CONTRACT_ID'].unique().tolist()
    ids_out_only = [c for c in ids_out_all if c not in ids_in_target]
    list_global_out_logs.extend(ids_out_only)
    
    report_summary.append({
        'WEEK_NUM': wk, 'TOTAL_EXPECTED': total_expected, 'IN_DICTIONARY': len(in_dict_list),
        'LOGGED_IN_TARGET': len(ids_in_target), 'LOGGED_OTHER_TARGETS': len(ids_out_only),
        'MISSING_IN_DICT': len(missing_from_dict), 'UNKNOWN_IDS': len(unknowns)
    })
    
    if not df_processed.empty:
        list_analysis_processed.append(df_processed.copy())

    df_verify_merge = pd.merge(df_4_valid, df_2_wk[['LOGGED_TAG']].drop_duplicates(), left_on='TAG_ID', right_on='LOGGED_TAG', how='left', indicator=True)
    df_never_logged = df_verify_merge[df_verify_merge['_merge'] == 'left_only'].copy()
    df_never_logged['WEEK_NUM'] = wk

    grp_cols = ['WEEK_NUM', 'TIME_SLOT', 'ZONE', 'CONTRACT_ID', 'ENTITY_NAME', 'WEEKLY_FREQUENCY', 'ADDRESS', 'REGION']
    active_grp = [c for c in grp_cols if c in df_never_logged.columns]
    df_grouped_logs = df_never_logged.groupby(active_grp)['TAG_ID'].apply(lambda x: sorted(list(set(x)))).reset_index()

    if not df_grouped_logs.empty:
        tags_expanded = pd.DataFrame(df_grouped_logs['TAG_ID'].tolist(), index=df_grouped_logs.index) 
        tags_expanded.columns = [f'TAG_{i+1}' for i in range(tags_expanded.shape[1])]
        df_never_logged_final = pd.concat([df_grouped_logs.drop(columns='TAG_ID'), tags_expanded], axis=1)
        df_never_logged_final = df_never_logged_final.drop_duplicates(subset=['CONTRACT_ID']).copy()
    else:
        df_never_logged_final = pd.DataFrame(columns=active_grp)
    
    list_missing_logs.append(df_never_logged_final)

    df_logged_target = pd.merge(df_processed, df_1, left_on='LOG_CODE', right_on='LOG_CODE', how='inner')
    df_unmatched_merge = pd.merge(df_logged_target, df_never_logged_final, on='CONTRACT_ID', how='right', indicator=True)

    df_final_unmatched = df_unmatched_merge[df_unmatched_merge['_merge'] == 'right_only'].copy()
    df_final_unmatched = df_final_unmatched.drop(columns=['_merge'], errors='ignore')
    df_final_unmatched = df_final_unmatched.dropna(axis=1, how='all')
    df_final_unmatched = df_final_unmatched.drop(columns=df_final_unmatched.filter(regex=r'^TAG_\d+').columns)
    list_unmatched_logic.append(df_final_unmatched)

    if 'ADDRESS_x' in df_processed.columns:
        df_processed = df_processed.rename(columns={'ADDRESS_x': 'ADDRESS'})

    df_merge_out = pd.merge(df_1[['LOG_CODE']], df_processed, left_on='LOG_CODE', right_on='LOG_CODE', how='right', indicator=True)
    df_out_detail = df_merge_out[df_merge_out['_merge'] == 'right_only'].copy()
    df_out_detail = df_out_detail.drop(columns=['_merge'], errors='ignore')

    if 'DEVICE_IMEI' in df_out_detail.columns:
        df_out_detail['DEVICE_IMEI'] = df_out_detail['DEVICE_IMEI'].astype(str).str.replace(r'\.0$', '', regex=True)
    
    df_out_detail['TIMESTAMP'] = pd.to_datetime(df_out_detail['TIMESTAMP'], errors='coerce')
    df_out_detail = df_out_detail.sort_values(by='TIMESTAMP', ascending=False)
    df_out_detail['TIMESTAMP'] = df_out_detail['TIMESTAMP'].dt.strftime('%Y-%m-%d')
    list_out_of_target.append(df_out_detail)

df_5 = pd.concat(list_analysis_processed, ignore_index=True) if list_analysis_processed else pd.DataFrame()
df_6 = pd.concat(list_unknown_entities, ignore_index=True) if list_unknown_entities else pd.DataFrame()
df_7 = pd.concat(list_not_in_dict, ignore_index=True) if list_not_in_dict else pd.DataFrame()
df_8 = pd.concat(list_unmatched_logic, ignore_index=True) if list_unmatched_logic else pd.DataFrame()
df_9 = pd.concat(list_missing_logs, ignore_index=True) if list_missing_logs else pd.DataFrame()
df_10 = pd.concat(list_out_of_target, ignore_index=True) if list_out_of_target else pd.DataFrame()

dict_short = df_3[['TAG_ID','ENTITY_NAME','CATEGORY_TYPE', 'CONTRACT_ID', 'REGION','ADDRESS']].drop_duplicates(subset=['TAG_ID'])
df_11 = pd.merge(df_2, dict_short, left_on='LOGGED_TAG', right_on='TAG_ID', how='inner')
df_12 = df_11[df_11['CATEGORY_TYPE'] != 'PRIMARY'].copy()
df_12 = df_12[df_12['LOG_CODE'].isin(df_1['LOG_CODE'].unique())]

df_13 = pd.merge(df_2, df_3[['TAG_ID']].drop_duplicates(), left_on='LOGGED_TAG', right_on='TAG_ID', how='left', indicator=True)
df_13 = df_13[df_13['_merge'] == 'left_only'].copy()
df_14 = df_13[df_13['LOG_CODE'].isin(df_1['LOG_CODE'])].copy()

df_performance = pd.DataFrame(report_summary)
if not df_performance.empty:
    df_performance['%_COVERAGE'] = ((df_performance['LOGGED_IN_TARGET'] / df_performance['TOTAL_EXPECTED']) * 100).round(0).fillna(0).astype(int)
    df_performance['STATUS_LIGHT'] = df_performance['%_COVERAGE'].apply(func_get_status)

if not df_8.empty:
    total_wks = df_2['WEEK_NUM'].nunique()
    count_absense = df_8.groupby('CONTRACT_ID')['WEEK_NUM'].nunique().reset_index()
    count_absense.columns = ['CONTRACT_ID', 'WEEKS_MISSING_COUNT']
    
    def func_critical_status(row):
        if row['WEEKS_MISSING_COUNT'] == total_wks: return "🔴 CRITICAL (Never Logged)"
        return "🟠 WARNING"

    count_absense['CRITICAL_LEVEL'] = count_absense.apply(func_critical_status, axis=1)
    df_8_final = df_8.drop_duplicates(subset=['CONTRACT_ID']).merge(count_absense, on='CONTRACT_ID', how='left')

if not df_5.empty: 
    df_calc = df_5.copy()
    df_calc['TIMESTAMP'] = pd.to_datetime(df_calc['TIMESTAMP']).dt.strftime('%Y-%m-%d')
    df_grouped_duplicates = df_calc.groupby(['CONTRACT_ID', 'LOGGED_TAG', 'TIMESTAMP'], dropna=False).agg({
        'LOGGED_TAG': 'count',
        'LOG_CODE': lambda x: ", ".join(sorted(list(set(x.dropna().astype(str)))))
    }).rename(columns={'LOGGED_TAG': 'LOG_COUNT', 'LOG_CODE': 'LOG_CODES_LIST'}).reset_index()
    df_15 = df_grouped_duplicates[df_grouped_duplicates['LOG_COUNT'] > 1].copy()

if list_all_inputs:
    df_all_in = pd.concat(list_all_inputs, ignore_index=True)
    df_base_pivot = pd.merge(df_2, df_3, left_on='LOGGED_TAG', right_on='TAG_ID', how='inner')
    df_base_pivot = pd.merge(df_base_pivot, df_all_in, on=['CONTRACT_ID', 'WEEK_NUM'], how='inner')
    
    pivot_cols = [c for c in ['CONTRACT_ID', 'ENTITY_NAME', 'OBJECT_TYPE', 'ZONE', 'ADDRESS', 'WEEKLY_FREQUENCY'] if c in df_base_pivot.columns]
    df_pivot_1 = df_base_pivot.pivot_table(index=pivot_cols, columns='WEEK_NUM', values='TIMESTAMP', aggfunc='nunique', fill_value=0).reset_index()
    df_pivot_1.columns = [f"WEEK_{int(col)}" if isinstance(col, (int, float)) else col for col in df_pivot_1.columns]
    
    wk_cols = [col for col in df_pivot_1.columns if re.match(r'^WEEK_\d+$', str(col))]
    df_pivot_res = df_pivot_1.copy()
    df_pivot_res['DIVISOR'] = pd.to_numeric(df_pivot_res['WEEKLY_FREQUENCY'], errors='coerce').fillna(1).replace(0, 1)

    for col in wk_cols:
        df_pivot_res[f"{col}_%"] = ((df_pivot_res[col] / df_pivot_res['DIVISOR'] * 100)).fillna(0).round(0).astype(int)
        df_pivot_res[f"STATUS_{col}"] = df_pivot_res[f"{col}_%"].apply(func_get_status)
        df_pivot_res[f"{col}_%"] = df_pivot_res[f"{col}_%"].astype(str) + '%'
    
    df_pivot_res = df_pivot_res.drop(columns=['DIVISOR'])
    pct_cols = [c for c in df_pivot_res.columns if 'WEEK' in c and '%' in c]
    df_math = df_pivot_res[pct_cols].replace(r'%', '', regex=True).apply(pd.to_numeric, errors='coerce').clip(upper=100)
    
    df_pivot_res['GLOBAL_RESULT_%'] = df_math.mean(axis=1).fillna(0).round(0).astype(int)
    df_pivot_res['GLOBAL_STATUS'] = df_pivot_res['GLOBAL_RESULT_%'].apply(func_get_status)
    df_pivot_res['GLOBAL_RESULT_%'] = df_pivot_res['GLOBAL_RESULT_%'].astype(str) + '%'

    meta_cols = [c for c in df_pivot_res.columns if c not in wk_cols and "%" not in str(c) and "STATUS" not in str(c)]
    final_order = meta_cols + ['GLOBAL_RESULT_%', 'GLOBAL_STATUS']
    for col in wk_cols:
        final_order.extend([col, f"{col}_%", f"STATUS_{col}"])
    df_pivot_final = df_pivot_res[final_order]

OUTPUT_DIR = "reports/generated/"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
export_path = os.path.join(OUTPUT_DIR, f"Consolidated_Report_{datetime.datetime.now().strftime('%Y%m%d')}.xlsx")

export_config = {
    1: (df_pivot_final, "PERFORMANCE_BY_ID", "GLOBAL PERFORMANCE METRICS BY ID"),
    2: (df_5, "DETAILED_LOGS", "CHRONOLOGICAL LOGGED EVENTS"),
    3: (df_performance, "WEEKLY_SUMMARY", "OVERALL WEEKLY PERFORMANCE"),
    4: (df_8_final if not df_8.empty else None, "CRITICAL_MISSING", "ENTITY CRITICALITY - NEVER LOGGED"),
    5: (df_10, "OUT_OF_TARGET_LOGS", "LOGS DETECTED OUTSIDE ASSIGNED CODES"),
    6: (df_15 if 'df_15' in locals() else None, "DUPLICATE_ENTRIES", "IDENTIFIED MULTIPLE LOGS PER DAY"),
    7: (df_14, "UNREGISTERED_TAGS", "TAGS LOGGED BUT MISSING FROM DICTIONARY")
}

with pd.ExcelWriter(export_path, engine='xlsxwriter', datetime_format='yyyy-mm-dd') as writer:
    workbook = writer.book
    fmt_title = workbook.add_format({'bold': True, 'font_size': 14, 'font_color': '#2E75B5'})
    fmt_head = workbook.add_format({'bold': True, 'bg_color': '#CFD8DC', 'border': 1})
    
    for key in sorted(export_config.keys()):
        df_obj, sheet_name, title_text = export_config[key]
        if df_obj is None or df_obj.empty: continue
        
        df_obj.to_excel(writer, sheet_name=sheet_name, index=False, startrow=2)
        sheet = writer.sheets[sheet_name]
        sheet.write('A1', title_text, fmt_title)
        
        for col_num, value in enumerate(df_obj.columns.values):
            sheet.write(2, col_num, value, fmt_head)
            sheet.set_column(col_num, col_num, 18)

print(f"Process completed. Report saved at: {export_path}")

In [ ]:
Questo codice realizza una pipeline ETL (Extract, Transform, Load) automatizzata per l'analisi avanzata di log operativi e il monitoraggio delle prestazioni (KPI).

È progettato per gestire flussi di dati complessi, incrociando anagrafiche fisse con aggiornamenti settimanali e log grezzi. Ecco i pilastri del processo:

Integrazione e Pulizia (Data Cleansing): Il sistema importa dati da diverse sorgenti (Excel e Parquet), standardizza i formati, normalizza i codici identificativi e pulisce le stringhe per garantire l'integrità dei dati prima dell'analisi.

Motore di Validazione (Quality Assurance): Esegue un check incrociato tra i log rilevati "sul campo" e i dizionari di riferimento. Identifica automaticamente anomalie come tag non censiti, letture fuori target geografico/logico e record duplicati nello stesso giorno.

Analisi Longitudinale (KPI Tracking): Attraverso operazioni di pivoting e raggruppamento, calcola la percentuale di "copertura" (rendimento) per ogni entità, sia su base settimanale che globale, assegnando uno stato (Semaforo) basato su soglie di performance personalizzabili.

Reporting Automatizzato: Genera un report Excel professionale multi-foglio. Ogni sezione è dedicata a una specifica criticità (es. entità mai rilevate, letture errate, analisi sospetti), formattata con titoli e layout pronti per essere presentati a livello di management (Business Intelligence).

In sintesi, trasforma una mole di dati grezzi e disordinati in uno strumento decisionale strutturato, evidenziando immediatamente dove intervenire per correggere inefficienze operative.